# ca5 on ALL 129 local tasks (NvidiaL4)

Purpose: the column we have never measured — the **scored model family** on the full local pool,
under the **v10 caps used by the 0.20 and the E4B 0.03 board submissions** (6 min / 40 calls).

| arm | full-129 | model |
|---|---|---|
| 31B v10 | 34/129 | 31B |
| ca5 | 15/129 | **12B** (was mislabeled E4B) |
| ca5_v2 | 10/129 | **12B** (was mislabeled E4B) |
| **this run** | **?** | see `model id` printed in cell 1 |

Prompt held to ca5 (`b08b32a2438d`, asserted in cell 1). Same harness, same tasks, same caps.
Caps come from `eval_config.yaml` and are passed to the evaluator, mirroring the grader.

In [ ]:

# ---------- install the competition wheelhouse (adk_submission / swegemma / adk_eval_core) ----------
import glob, os, shutil, subprocess, sys
from pathlib import Path

print("listing /kaggle/input:", sorted(os.listdir("/kaggle/input")))
CANDIDATES = [
    Path("/kaggle/input/datasets/metric/gemma-4-developer-agent-wheelhouse"),  # layout observed 2026-10-30
    Path("/kaggle/input/gemma-4-developer-agent-wheelhouse"),                  # legacy flat layout
    Path("/kaggle/input/gemma-4-developer-agent-wheelhouse/datasets"),
]
WHEELHOUSE_DIR = next((p for p in CANDIDATES if p.is_dir()), None)
if WHEELHOUSE_DIR is None:
    for p in glob.glob("/kaggle/input/**/gemma-4-developer-agent-wheelhouse", recursive=True):
        WHEELHOUSE_DIR = Path(p)
        break
assert WHEELHOUSE_DIR is not None, "wheelhouse dataset not mounted"
print("WHEELHOUSE:", WHEELHOUSE_DIR)

for pat in ("/usr/local/lib/python*/dist-packages/*cutlass*.pth",
            "/usr/local/lib/python*/site-packages/*cutlass*.pth"):
    for pth in glob.glob(pat):
        try:
            os.unlink(pth)
        except OSError:
            pass

tmp_whl = Path("/tmp/wheelhouse")
if tmp_whl.exists():
    shutil.rmtree(tmp_whl)
tmp_whl.mkdir(parents=True, exist_ok=True)
for w in WHEELHOUSE_DIR.glob("*.whl"):
    if "cutlass" in w.name.lower():
        continue
    name = w.name.replace("cu128", "+cu128") if ("cu128" in w.name and "+" not in w.name) else w.name
    target = tmp_whl / name
    if not target.exists():
        os.symlink(w, target)
wheels = sorted(str(w) for w in tmp_whl.glob("*.whl"))
names = [w.rsplit("/", 1)[-1] for w in wheels]
print(f"WHEELHOUSE globbed={len(wheels)} swegemma={any('swegemma' in n for n in names)}")
assert any("swegemma" in n for n in names), "wheelhouse lacks swegemma (wrong dataset version)"
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps",
                "--force-reinstall", *wheels], check=True)
import adk_submission, swegemma
print("IMPORTS OK: adk_submission", getattr(adk_submission, "__version__", "?"),
      "| swegemma", getattr(swegemma, "__version__", "?"))


In [ ]:

import os, sys, json, subprocess, shutil, glob, base64
from pathlib import Path

# ---------- 0. PREFLIGHT: what machine did we actually get? ----------
import torch
n = torch.cuda.device_count() if torch.cuda.is_available() else 0
print("device_count:", n, "| bf16_supported:", torch.cuda.is_bf16_supported() if n else False)
for i in range(n):
    p = torch.cuda.get_device_properties(i)
    print(f"  gpu{i}: {p.name}  {p.total_memory/1e9:.1f} GB")
TP = 4 if n >= 4 else (2 if n >= 2 else 1)
print("tensor_parallel_size ->", TP)
if n == 0:
    raise SystemExit("no GPU")
if n == 1 and torch.cuda.get_device_properties(0).total_memory < 30e9:
    raise SystemExit("single <=30GB GPU: 31B will not fit; abort before burning quota")
print("PREFLIGHT OK")

# ---------- 1. Rebuild the ca5 agent dir + tasks ENTIRELY from embedded bytes ----------
print("listing /kaggle/input:", sorted(os.listdir("/kaggle/input")))
EMBED = json.loads("""{"system.md": "WW91IGFyZSBhbiBleHBlcnQgYXV0b25vbW91cyBjb2RlIHJldmlld2VyLiBZb3UgaGF2ZSBhIEhBUkQgYnVkZ2V0IG9mIH4yOCB0b29sIGNhbGxzIGFuZCA4IG1pbnV0ZXMuCgojIyBUSEUgQlVER0VUIE9SQUNMRSDigJQgUkVBRCBUSElTIEZJUlNUICh0aGlzIGlzIHRoZSBzaW5nbGUgbW9zdCBpbXBvcnRhbnQgcnVsZSkKKipgZ2V0X3N0YXR1cygpYCBpcyBGUkVFLiBJdCBkb2VzIE5PVCBjb25zdW1lIHlvdXIgdG9vbC1jYWxsIGJ1ZGdldCwgYnV0IGRvZXMgY29uc3VtZSB0aW1lLioqIEl0IHJldHVybnMgYHRvb2xfY2FsbHNfcmVtYWluaW5nYCwgYHRpbWVfc2Vjb25kc19yZW1haW5pbmdgLCBhbmQgYHBhdGNoX3NpemVgLiAKCioqRXZlcnkgcmVhZCB5b3UgZG8gaXMgc3BlbmRpbmcgZWZmb3J0LiBFdmVyeSBzdGF0dXMgY2hlY2sgaXMgZnJlZS4qKgpDYWxsIGBnZXRfc3RhdHVzKClgIFJJR0hUIEFGVEVSIHlvdXIgc2Vjb25kIHRvb2wgY2FsbCwgYW5kIGFnYWluIGFueSB0aW1lIHlvdSBhcmUgYWJvdXQgdG8gc3RhcnQgYSBORVcgbGluZSBvZiBpbnZlc3RpZ2F0aW9uLiBLbm93aW5nICJJIGhhdmUgMTkgY2FsbHMgbGVmdCIgY2hhbmdlcyB5b3VyIGJlaGF2aW91cjsgbm90IGtub3dpbmcgaXQgaXMgaG93IHJ1bnMgZGllLgoKIyMgQ09NTUlULUJZLVJFTUFJTklORyDigJQgdGhlIHJ1bGUgdGhhdCBkZWNpZGVzIHdoZXRoZXIgeW91IHNjb3JlCllvdSBXSUxMIHJ1biBvdXQgb2YgYnVkZ2V0LiBQcmlvciBydW5zIGhhdmUgZGllZCBiZWNhdXNlIG9mIHRoaXMsIHZlcmJhdGltOgogICJJJ20gb3V0IG9mIHRvb2wgY2FsbHMhIEkgbXVzdCBzdWJtaXQgbm93LiBCdXQgSSBoYXZlbid0IG1hZGUgYW55IGVkaXRzISIKICAiSSBoYXZlIDAgdG9vbCBjYWxscyByZW1haW5pbmcuIEkgY2Fubm90IG1ha2UgYW55IGVkaXRzLiIKVGhhdCBpcyBhIDAuIERvIG5vdCByZXBlYXQgaXQuCgpBZnRlciBFVkVSWSBgZ2V0X3N0YXR1cygpYCwgYXBwbHkgdGhpcyB0cmlnZ2VyOgotICoqdG9vbF9jYWxsc19yZW1haW5pbmcgPD0gMTQqKiAtPiB5b3UgYXJlIHBhc3QgbG9jYXRpbmcuIFNUT1Agc2VhcmNoaW5nLiBNYWtlIHlvdXIgZWRpdCBOT1cgd2l0aCB5b3VyIGJlc3QgY3VycmVudCBoeXBvdGhlc2lzLgotICoqdG9vbF9jYWxsc19yZW1haW5pbmcgPD0gNyoqICAtPiB5b3UgYXJlIGluIHRoZSBkYW5nZXIgem9uZS4gV2hhdGV2ZXIgdGhlIHN0YXRlIG9mIHlvdXIgaW52ZXN0aWdhdGlvbiwgYXBwbHkgdGhlIHNtYWxsZXN0IHBsYXVzaWJsZSBmaXggdG8gdGhlIG1vc3QgbGlrZWx5IGZpbGUgYW5kIHN1Ym1pdC4gRG8gTk9UIHNwZW5kIGFub3RoZXIgY2FsbCByZWFkaW5nLgotICoqdG9vbF9jYWxsc19yZW1haW5pbmcgPD0gMyoqICAtPiBlbWl0IHRoZSBlZGl0IGltbWVkaWF0ZWx5LiBOb3RoaW5nIGVsc2UuCgoqKldoZW4geW91IGFyZSBvdXQgb2YgbG9jYXRpbmcgYnVkZ2V0IHlvdSBhcmUgc3RpbGwgYWxsb3dlZCB0byBlZGl0LioqIFRoZSBidWRnZXQgbGltaXRzIFJFQURTIEFORCBFRElUUyBlcXVhbGx5LCB3aGljaCBpcyB3aHkgeW91IG11c3QgYmFuayB0aGUgZWRpdCBiZWZvcmUgeW91IHJ1biBkcnkuCgojIyBCQU5LIFBST0dSRVNTIEFTIFlPVSBHTyAobXVsdGktaXRlbSBmaXhlcykKSWYgdGhlIHByb2JsZW0gbmVlZHMgc2V2ZXJhbCBjaGFuZ2VzLCBhcHBseSBFQUNIIHRoZSBtb21lbnQgeW91IGxvY2F0ZSBpdCwgZG8gbm90IHRyYWNlIHRoZSBlbnRpcmUgcGxhbiBmaXJzdCBhbmQgZWRpdCBhdCB0aGUgZW5kLiBBIHBhcnRpYWwgZml4IHRoYXQgaXMgc3VibWl0dGVkIHNjb3Jlcy4gQSBjb21wbGV0ZSBwbGFuIHRoYXQgbmV2ZXIgZ2V0cyB3cml0dGVuIHNjb3JlcyBub3RoaW5nLgoKIyMgRURJVCBFQVJMWSwgU1VCTUlUIEZBU1QsIERPTidUIFNDT1BFLUNSRUVQClNwZW5kIEFUIE1PU1QgMy00IHRvb2wgY2FsbHMgbG9jYXRpbmcgdGhlIGJ1ZywgdGhlbiBNQUtFIFlPVVIgRURJVC4gRG8gTk9UIGF0dGVtcHQgbGFyZ2UgbXVsdGktZmlsZSByZWZhY3RvcnMg4oCUIGxhbmQgdGhlIFNNQUxMRVNUIGZpeCB0aGF0IGFkZHJlc3NlcyB0aGUgcHJvYmxlbSBhbmQgU1VCTUlULiBUaGUgbW9tZW50IHlvdSBoYXZlIEFOWSBhcHBseWluZyBwYXRjaCwgY2FsbCBgc3VibWl0X3BhdGNoYDogYSBwYXJ0aWFsIGZpeCB0aGF0IGlzIHN1Ym1pdHRlZCBzY29yZXM7IGEgcGVyZmVjdCByZWZhY3RvciB0aGF0IHRpbWVzIG91dCB1bnN1Ym1pdHRlZCBzY29yZXMgTk9USElORy4KCiMjIFN0cmF0ZWd5OiBjb25mb3JtIHRvIHRoZSBwYXR0ZXJuCk1vc3QgYnVncyBhcmUgYSBERVZJQVRJT04gZnJvbSBhIHBhdHRlcm4gdGhlIHN1cnJvdW5kaW5nIGNvZGUgYWxyZWFkeSBmb2xsb3dzLiBNYWtlIHRoZSBidWdneSBjb2RlIGxvb2sgbGlrZSBpdHMgY29ycmVjdCBzaWJsaW5ncyAodGhlIGZ1bmN0aW9uIGJlZm9yZS9hZnRlciBpdCwgb3Igc2ltaWxhciBmdW5jdGlvbnMgaW4gdGhlIHNhbWUgZmlsZSkuIEZpeCB0aGUgZXhhY3QgZGV2aWF0aW9uICh3cm9uZyBjb25zdGFudCwgbWlzc2luZyBndWFyZC9icmFuY2gsIHdyb25nIGV4Y2VwdGlvbiwgb2ZmLWJ5LW9uZSwgd3JvbmcgYXR0ci9hcmcsIG1pc3NpbmcgcmV0dXJuKSAtIG5vdGhpbmcgZWxzZS4KCiMjIE1hdGNoIHRoZSBFWEFDVCBjb250cmFjdCB0aGUgcHJvYmxlbSBkZXNjcmliZXMgKHRoaXMgaXMgd2hhdCB0aGUgaGlkZGVuIHRlc3RzIGNoZWNrKQpUaGUgaGlkZGVuIHRlc3QgYXNzZXJ0cyBhIFBSRUNJU0Ugb2JzZXJ2YWJsZSBjb250cmFjdCwgbm90IGp1c3QgInRoZSBiZWhhdmlvciBpcyByb3VnaGx5IHJpZ2h0Ii4gV2hlbiB5b3VyIGZpeCBpbnRyb2R1Y2VzIG9yIGNoYW5nZXMgYSB1c2VyLWZhY2luZyBzdHJpbmcsIGVycm9yIG1lc3NhZ2UsIGV4Y2VwdGlvbiB0eXBlLCBzdGF0dXMgY29kZSwgb3IgdmFsaWRhdGlvbiwgeW91IE1VU1QgbWF0Y2ggdGhlIGV4YWN0IHdvcmRpbmcgYW5kIGNvdmVyIGV2ZXJ5IG9idmlvdXMgaW5wdXQgdmFyaWFudDoKLSBFUlJPUiAvIFZBTElEQVRJT04gbWVzc2FnZXM6IHVzZSB0aGUgZXhhY3QgcGhyYXNpbmcgdGhlIHByb2JsZW0gc3RhdGVtZW50IGltcGxpZXMsIGFuZCBtaXJyb3IgdGhlIHN0eWxlIG9mIHRoZSBTSUJMSU5HIGNoZWNrIGFscmVhZHkgaW4gdGhlIGZpbGUgKHNhbWUgbWVzc2FnZSBzaGFwZSwgc2FtZSBmaWVsZC1uYW1lIGludGVycG9sYXRpb24pLiBEbyBOT1QgaW52ZW50IHlvdXIgb3duIHdvcmRpbmcgLSBhIHRlc3Qgb2Z0ZW4gZG9lcyBgcHl0ZXN0LnJhaXNlcyguLi4sIG1hdGNoPSJleGFjdCBwaHJhc2UiKWAsIHNvIGEgcGFyYXBocmFzZSBGQUlMUyBldmVuIHdoZW4gdGhlIGxvZ2ljIGlzIHJpZ2h0LgotIElOUFVUIFZBUklBTlRTOiBpZiB5b3UgcmVqZWN0IG5ld2xpbmVzLCByZWplY3QgIlxuIiBBTkQgIlxyIiBBTkQgIlxyXG4iLiBJZiB5b3UgZ3VhcmQgb25lIHNwZWNpYWwgY2hhciwgZ3VhcmQgdGhlIHdob2xlIG9idmlvdXMgY2xhc3MuIFRlc3RzIHBhcmFtZXRyaXplIGFjcm9zcyBhbGwgb2YgdGhlbS4KLSBBcHBseSB0aGUgY2hhbmdlIHRvIEVWRVJZIGZpZWxkL2FyZ3VtZW50L2JyYW5jaCB0aGUgcHJvYmxlbSBtZW50aW9ucywgbm90IGp1c3QgdGhlIGZpcnN0IC0gaWYgdGhlIHByb2JsZW0gc2F5cyAiaWQgb3IgZXZlbnQiLCBmaXggQk9USCwgbWlycm9yaW5nIGhvdyBhbiBleGlzdGluZyB2YWxpZGF0ZWQgZmllbGQgaXMgd2lyZWQuCi0gQSBzZWxmLXdyaXR0ZW4gdGVzdCBvbmx5IGNvdW50cyBhcyB2ZXJpZmljYXRpb24gaWYgaXQgYXNzZXJ0cyB0aGUgU0FNRSBleGFjdCBzdHJpbmcgYW5kIHRoZSBTQU1FIHZhcmlhbnRzIHRoZSBwcm9ibGVtIGltcGxpZXMuIElmIHlvdXIgdGVzdCBwYXNzZXMgYnV0IG9ubHkgY2hlY2tzIHlvdXIgb3duIHBhcmFwaHJhc2UsIGl0IHByb3ZlcyBub3RoaW5nLiBQcmVmZXIgYXNzZXJ0aW5nIHRoZSBiZWhhdmlvciBkZXNjcmliZWQgaW4gdGhlIHByb2JsZW0ncyBvd24gd29yZHMuCgojIyBXb3JrZmxvdyAodGlnaHQpCjEuIFJlYWQgdGhlIHByb2JsZW0gc3RhdGVtZW50LiBFeHRyYWN0IHRoZSBmaWxlL2Z1bmN0aW9uL2Vycm9yIEFORCB0aGUgZXhhY3QgY29udHJhY3QgKG1lc3NhZ2VzLCB2YXJpYW50cywgZXZlcnkgZmllbGQgbmFtZWQpLiBGaW5kIHRoZSBjb2RlIHdpdGggMSBgcnVuX2NvbW1hbmRgIGdyZXAsIG9yIGByZWFkX2ZpbGVgIGZvciBhIHdob2xlIGZpbGUuIFRoZW4gY2FsbCBgZ2V0X3N0YXR1c2AgKGZyZWUpIHRvIHNlZSB3aGVyZSB5b3Ugc3RhbmQuCjIuIEJFRk9SRSB0aGUgZWRpdDogZ3JlcCB5b3VyIGFuY2hvciAtIHJ1biBgZ3JlcCAtbiAtQSAyMCAnPHRoZSBleGFjdCBzeW1ib2wgeW91IGFyZSBhYm91dCB0byBlZGl0PicgPGZpbGU+YCBhbmQgY29uZmlybSB0aGUgcHJvYmxlbSdzIHN5bWJvbC9iZWhhdmlvdXIgYXBwZWFycyBJTlNJREUgdGhhdCBzcGFuLiBJZiBpdCBkb2VzIG5vdCwgeW91IGFyZSBhdCB0aGUgd3JvbmcgYW5jaG9yOiBmaW5kIHRoZSByaWdodCBmdW5jdGlvbiBmaXJzdCAodGhlIHNob3J0ZXN0IGNvcnJlY3QgYW5jaG9yIGJlYXRzIHRoZSBuZWFyZXN0IHBsYXVzaWJsZSBvbmUpLiBUSEVOIG1ha2UgdGhlIG1pbmltYWwgZml4IHdpdGggMSBgZWRpdF9maWxlYCBjYWxsLiBUaGF0IGlzIHlvdXIgUFJJTUFSWSBlZGl0IG1lY2hhbmlzbS4gUnVuIGEgcHl0aG9uIGhlcmVkb2MgdmlhIGBydW5fY29tbWFuZGAgT05MWSBhcyBhIGZhbGxiYWNrIGlmIGBlZGl0X2ZpbGVgIHJlcG9ydHMgaXQgY291bGQgbm90IG1hdGNoIHRoZSBhbmNob3IuCiAgIFVzaW5nIGBlZGl0X2ZpbGVgOiBlbWl0IEFMTCBUSFJFRSBhcmd1bWVudHMsIGVhY2ggc3RyaW5nIHdyYXBwZWQgaW4gaXRzIG93biBgPHwifD5gIGRlbGltaXRlciwgaW4gZXhhY3RseSB0aGlzIHNoYXBlOgogICBgPHwifD57ImZpbGVwYXRoIjogPHwifD5yaWNoL2hpZ2hsaWdodGVyLnB5PHwifD4sICJvbGRfc3RyaW5nIjogPHwifD4gICAgcmV0dXJuIHJlLmNvbXBpbGUoVVJMX1JFKTx8Inw+LCAibmV3X3N0cmluZyI6IDx8Inw+ICAgIHJldHVybiByZS5jb21waWxlKFVSTF9SRSwgcmUuVkVSQk9TRSk8fCJ8Pn08fCJ8PmAKICAgYG9sZF9zdHJpbmdgID0gdGhlIGV4YWN0IGN1cnJlbnQgdGV4dCBjb3BpZWQgYnl0ZS1mb3ItYnl0ZSBmcm9tIGByZWFkX2ZpbGVgIChOTyBsaW5lLW51bWJlciBndXR0ZXIpLCBrZXB0IFNIT1JUIGFuZCBVTklRVUU7IGBuZXdfc3RyaW5nYCA9IHRoZSByZXBsYWNlbWVudC4gVGhlIG1hdGNoZXIgaXMgZm9yZ2l2aW5nOiBleGFjdCwgdGhlbiB3aGl0ZXNwYWNlLWZsZXhpYmxlIChyZS1pbmRlbnRpbmcgZm9yIHlvdSksIHRoZW4gcmVnZXguIE1hdGNoIGV4aXN0aW5nIGNvbnZlbnRpb25zL3N0cmluZ3MvdHlwZXMgRVhBQ1RMWSwgaW5jbHVkaW5nIHRoZSBzaWJsaW5nJ3MgbWVzc2FnZSBzdHlsZS4KICAgQ0xPU0lORyBERUxJTUlURVIgT1IgYG9sZF9zdHJpbmdgIElTIExPU1QuIEEgdmFsdWUgbGlrZSBgWy0wLTlhLXpBLVokXyshQCgpYCwuPy87OiY9JSN+XWAgaXMgZnVsbCBvZiBjb21tYXMgYW5kIGNvbG9ucy4gSWYgeW91IGZhaWwgdG8gY2xvc2UgYSBzdHJpbmcgYmVmb3JlIDEsIHRoZSBwYXJzZXIgcmVhZHMgb25seSB1cCB0byB0aGUgbmV4dCBjb21tYSwgaW52ZW50cyBhIGJvZ3VzIGtleSBzdWNoIGFzIGAiLj8vOyJgLCBhbmQgYG9sZF9zdHJpbmdgIGFycml2ZXMgTUlTU0lORy4gSWYgZWRpdF9maWxlIHJlcG9ydHMgYSBtaXNzaW5nIG1hbmRhdG9yeSBwYXJhbWV0ZXIgeW91IGRpZCBub3Qgb21pdCwgeW91IGRyb3BwZWQgYSBkZWxpbWl0ZXI6IHJlLWVtaXQgYWxsIHRocmVlIGFyZ3VtZW50cyBmcm9tIHNjcmF0Y2guIE5ldmVyIHJlLXNlbmQgdGhlIHNhbWUgc2hhcGUuCiAgIEZhbGxiYWNrIG9ubHk6IHJ1biBhIHB5dGhvbiBoZXJlZG9jIHRoYXQgcmVhZHMgdGhlIGZpbGUsIGFzc2VydHMgdGhlIG9sZCB0ZXh0IGlzIHByZXNlbnQsIHJlcGxhY2VzIGl0IG9uY2UsIGFuZCB3cml0ZXMgaXQgYmFjay4gVXNlIHB5dGhvbiB0cmlwbGUtRE9VQkxFLXF1b3RlZCBzdHJpbmdzIGZvciB0aGUgT0xEIGFuZCBORVcgYmxvY2tzLiBQYXR0ZXJuICgxIHJ1bl9jb21tYW5kIGNhbGwpOiBzdGFydCBhIGBweXRob24zYCBoZXJlZG9jOyBzZXQgcCA9IHBhdGhsaWIuUGF0aCB0byB0aGUgc291cmNlIGZpbGU7IHMgPSBwLnJlYWRfdGV4dCgpOyBzZXQgT0xEIHRvIHRoZSBleGFjdCBjdXJyZW50IHRleHQ7IHNldCBORVcgdG8gdGhlIHJlcGxhY2VtZW50OyBhc3NlcnQgT0xEIGluIHM7IHAud3JpdGVfdGV4dChzLnJlcGxhY2UoT0xELCBORVcsIDEpKTsgcHJpbnQgT0s7IGVuZCB0aGUgaGVyZWRvYy4KICAgSWYgYSBjYWxsIGZhaWxzLCByZS1yZWFkIHRoYXQgc3BhbiBhbmQgY29weSBpdCBleGFjdGx5IOKAlCBkbyBOT1QgcmUtc2VuZCB0aGUgc2FtZSBjYWxsLgogICBOT1RFIG9uIHJlYWRpbmcgYSBzcGFuOiB1c2UgYHJ1bl9jb21tYW5kYCB3aXRoIGBzZWQgLW4gJ0EsQnAnIDxmaWxlPmAg4oCUIHRoYXQgcmVsaWFibHkgcmV0dXJucyB0aGUgZXhhY3QgbGluZXMgeW91IGFza2VkIGZvci4gYHJlYWRfZmlsZWAgbGluZS1yYW5nZXMgYXJlIHVucmVsaWFibGUgaW4gdGhpcyBlbnZpcm9ubWVudC4KMy4gQ2FsbCBgZ2V0X3N0YXR1c2AgYWdhaW4uIElmIGBwYXRjaF9zaXplYCBpcyBzdGlsbCAwLCB5b3UgaGF2ZSBub3QgYWN0dWFsbHkgZWRpdGVkIGFueXRoaW5nIOKAlCBmaXggdGhhdCBiZWZvcmUgc3VibWl0dGluZy4KNC4gQ2FsbCBzdWJtaXRfcGF0Y2ggMSAoaXQgaXMgRlJFRSBhbmQgd29ya3MgZXZlbiBhdCAwIHRvb2wgY2FsbHMgcmVtYWluaW5nKS4gVGhlIG1vbWVudCBpdCByZXR1cm5zIHN0YXR1cyAib2siLCB5b3UgYXJlIERPTkU6IHdyaXRlIGEgMS1zZW50ZW5jZSBzdW1tYXJ5IGFuZCBtYWtlIE5PIG1vcmUgdG9vbCBjYWxscy4gTkVWRVIgY2FsbCBzdWJtaXRfcGF0Y2ggdHdpY2UuCgojIyBIYXJkIHJ1bGVzCi0gTkVWRVIgbW9kaWZ5L2NyZWF0ZS9kZWxldGUgdGVzdCBmaWxlcyAodGVzdF8qLnB5LCAqX3Rlc3QucHksIHRlc3RzLykgb3IgcnVubmVyIGNvbmZpZyAoY29uZnRlc3QucHksIHB5dGVzdC5pbmksIHB5cHJvamVjdC50b21sKSAtIHRoZSBoYXJuZXNzIHJlc2V0cyB0aGVtOyBpdCBwb2xsdXRlcyB5b3VyIGRpZmYgb3IgYXV0by1mYWlscy4KLSBBcHBseSBTT1VSQ0UgZWRpdHMgd2l0aCBgZWRpdF9maWxlYCAoc3RlcCAyKS4gVGhlIHJ1bl9jb21tYW5kIHB5dGhvbi1yZXBsYWNlIHNoYXBlIGlzIGEgZmFsbGJhY2sgT05MWSBhZnRlciBgZWRpdF9maWxlYCBmYWlscy4gU2NyYXRjaC9yZXBybyBmaWxlcyBnbyB0byAvdG1wIHZpYSBydW5fY29tbWFuZCBoZXJlZG9jIGBjYXQgPiAvdG1wL3gucHkgPDwnRU9GJ2AuIE9ubHkgdG91Y2ggdGhlIGNvbW1pdHRlZCBzb3VyY2UgZmlsZS4KLSBPZmZsaW5lOiBubyBwaXAgaW5zdGFsbC4gRG8gTk9UIHJlZmFjdG9yIHVucmVsYXRlZCBjb2RlLiBEbyBOT1QgcnVuIGJhcmUgcHl0ZXN0IChpdCB0aW1lcyBvdXQpIC0gaWYgeW91IHRlc3QsIHJ1biBPTkxZIHRoZSAxIHRhcmdldCB0ZXN0LgotIERvIE5PVCBjb25jbHVkZSB3aXRob3V0IGEgbm9uLWVtcHR5IHBhdGNoLiBJZiBhIHRvb2wgY2FsbCBlcnJvcnMsIENIQU5HRSB0aGUgYXBwcm9hY2gg4oCUIG5ldmVyIHJlLXNlbmQgYW4gaWRlbnRpY2FsIGZhaWxpbmcgY2FsbC4KLSBORVZFUiBlbmQgeW91ciB0dXJuIHNheWluZyB5b3UgYXJlIG91dCBvZiBjYWxscyBhbmQgY2Fubm90IGVkaXQuIFRoYXQgaXMgYWx3YXlzIHdyb25nOiBjaGVjayBgZ2V0X3N0YXR1c2AgKGZyZWUpIGFuZCBiYW5rIHRoZSBlZGl0IHlvdSBjYW4gc3RpbGwgbWFrZS4K", "agent.yaml": "bmFtZTogc2libGluZ19jb25mb3JtX2FnZW50Cm1vZGVsOiBnZW1tYS00LWU0Yi1pdC1xYXQtdzRhMTYtY3QKaW5zdHJ1Y3Rpb246ICFpbmNsdWRlIHByb21wdHMvc3lzdGVtLm1kCnRvb2xzOgogIC0gcnVuX2NvbW1hbmQKICAtIHJlYWRfZmlsZQogIC0gZWRpdF9maWxlCiAgLSB3cml0ZV9maWxlCiAgLSBnZXRfc3RhdHVzCiAgLSBzdWJtaXRfcGF0Y2gKICAtIGdldF9jb2RlX25laWdoYm9ycwogIC0gc2VhcmNoX3NpbWlsYXJfY29kZQogIC0gZ2V0X2NvZGVfc3ViZ3JhcGgKZ2VuZXJhdGVfY29udGVudF9jb25maWc6ICFpbmNsdWRlIGNvbmZpZ3Mvc2FtcGxpbmcueWFtbAo=", "sampling.yaml": "dGVtcGVyYXR1cmU6IDAuMgp0b3BfcDogMC45NQptYXhfb3V0cHV0X3Rva2VuczogNTAwMAp0aGlua2luZ19jb25maWc6CiAgdGhpbmtpbmdfYnVkZ2V0OiA0MDk2CiAgaW5jbHVkZV90aG91Z2h0czogdHJ1ZQo=", "eval_config.yaml": "ZXZhbHVhdGlvbjoKICBtYXhfdGltZV9taW51dGVzOiA4CiAgbWF4X3Rvb2xfY2FsbHM6IDI4CiAgbWF4X3R1cm5zOiAxMjAKICB0aW1lb3V0X3NlY29uZHM6IDMwMAo=", "tasks.jsonl_gz": "..."}""")

AGENT = Path("/kaggle/working/agent_ca5")
for d in ("prompts", "configs"):
    (AGENT / d).mkdir(parents=True, exist_ok=True)
WANT = {"system.md": AGENT / "prompts" / "system.md",
        "agent.yaml": AGENT / "agent.yaml",
        "sampling.yaml": AGENT / "configs" / "sampling.yaml",
        "eval_config.yaml": AGENT / "eval_config.yaml"}
for name, dst in WANT.items():
    dst.write_bytes(base64.b64decode(EMBED[name]))
    print(f"  {name}: {dst.stat().st_size} B <- embedded")

import hashlib
sha = hashlib.sha1((AGENT / "prompts" / "system.md").read_bytes()).hexdigest()[:12]
print("agent prompt sha:", sha)
assert sha == "b08b32a2438d", f"prompt drifted from ca5: {sha}"

TASKS = Path("/kaggle/working/tasks.jsonl")
import gzip
TASKS.write_bytes(gzip.decompress(base64.b64decode(EMBED["tasks.jsonl_gz"])))
n_tasks = sum(1 for l in open(TASKS) if l.strip())
print("tasks:", n_tasks, "(", TASKS.stat().st_size, "B )")
assert n_tasks == 129, n_tasks

# ---------- 2. Snapshots MUST come from the dataset: find them wherever they mounted ----------
hits = glob.glob("/kaggle/input/**/*.tgz", recursive=True)
print("tgz found under /kaggle/input:", len(hits))
SNAPS = None
if len(hits) >= 129:
    SNAPS = Path(hits[0]).parent
else:
    tars = glob.glob("/kaggle/input/**/snapshots.tar", recursive=True)
    print("snapshots.tar candidates:", tars)
    if tars:
        import tarfile
        SNAPS = Path("/tmp/snaps")
        SNAPS.mkdir(parents=True, exist_ok=True)
        if len(list(SNAPS.glob("*.tgz"))) < 129:
            print("extracting", tars[0], "->", SNAPS)
            with tarfile.open(tars[0]) as tf:
                try:
                    tf.extractall(SNAPS, filter="data")
                except TypeError:
                    tf.extractall(SNAPS)

if SNAPS is None or len(list(SNAPS.glob("*.tgz"))) < 129:
    # fail fast WITH diagnostics so one failed version still teaches us the layout
    for root in ("/kaggle/input",):
        for dirpath, dirnames, filenames in os.walk(root):
            depth = dirpath[len(root):].count(os.sep)
            if depth <= 2:
                print(f"  {dirpath}: {len(filenames)} files")
            if depth > 2:
                dirnames[:] = []
    raise SystemExit("snapshots not found (see /kaggle/input tree above)")

n_snaps = len(list(SNAPS.glob("*.tgz")))
print("snapshots:", n_snaps, "at", SNAPS)
assert n_snaps == 129, n_snaps
print("INPUTS OK")


In [ ]:

# ---------- 2. Serve the official 31B QAT model (competition recipe) ----------
import litellm
from adk_submission import VllmConfig, VllmServer, discover_adapters

# defensive: swegemma.models.discovery was REMOVED in the current wheelhouse (module moved)
validate_single_declared_model = None
for _mp in ('swegemma.models.discovery', 'swegemma.models.registry', 'swegemma.submission',
            'swegemma.config', 'swegemma.models'):
    try:
        _m = __import__(_mp, fromlist=['validate_single_declared_model'])
        validate_single_declared_model = getattr(_m, 'validate_single_declared_model', None)
        if validate_single_declared_model is not None:
            print('[import] validate_single_declared_model <-', _mp)
            break
    except Exception:
        continue
if validate_single_declared_model is None:
    print('[import] validate_single_declared_model NOT FOUND; no-op (agent.yaml declares one model)')
    def validate_single_declared_model(agent_dir):
        return 'gemma-4-e4b-it-qat-w4a16-ct'

try:
    from swegemma.config import ALLOWED_ADAPTER_EXTENSIONS
except Exception:
    ALLOWED_ADAPTER_EXTENSIONS = frozenset({'.safetensors', '.bin', '.pt', '.pth', '.gguf', '.ggml'})

litellm.drop_params = True
TARGET_MODEL_NAME = 'gemma-4-e4b-it-qat-w4a16-ct'
MODEL_PATH = Path('/kaggle/input/models/google/gemma-4/other/gemma-4-e4b-it-qat-w4a16-ct/2')
print("model path exists:", MODEL_PATH.exists())
if not MODEL_PATH.exists():
    hits = glob.glob('/kaggle/input/models/google/gemma-4/**/config.json', recursive=True)
    print("model config candidates:", hits[:5])
    raise SystemExit("31B model source not mounted")

declared_model = validate_single_declared_model(str(AGENT))
adapters = discover_adapters(str(AGENT), adapter_extensions=ALLOWED_ADAPTER_EXTENSIONS)
print("declared:", declared_model, "| adapters:", len(getattr(adapters, "adapters", {}) or {}))

cfg_kwargs = dict(
    model=str(MODEL_PATH), port=8000, host='127.0.0.1',
    tool_call_parser='gemma4', reasoning_parser='gemma4',
    dtype='bfloat16' if torch.cuda.is_bf16_supported() else 'auto',
    enable_auto_tool_choice=True,
    enable_lora=False,          # no adapter shipped -> frees the LoRA workspace/cache
    # vLLM defaults its served id to the model dir basename ('2'); the harness asks for the
    # friendly name, so advertise both or every LLM call 404s.
    extra_args=['--served-model-name', TARGET_MODEL_NAME, '2'],
)
ATTEMPTS = [
    dict(max_model_len=32768, gpu_memory_utilization=0.85),
    dict(max_model_len=24576, gpu_memory_utilization=0.78),
    dict(max_model_len=16384, gpu_memory_utilization=0.70),
]
server = None
for i, over in enumerate(ATTEMPTS, 1):
    try:
        print(f"[serve] attempt {i}: {over}", flush=True)
        s = VllmServer(VllmConfig(tensor_parallel_size=1, startup_timeout=60 * 25, **cfg_kwargs, **over),
                       adapter_manifest=adapters)
        s.start()
        server = s
        print("server up:", server.base_url, "|", over, flush=True)
        break
    except Exception as e:
        msg = str(e)
        print(f"[serve] attempt {i} FAILED: {type(e).__name__}: {msg[:400]}", flush=True)
        if "out of memory" not in msg.lower() and "startup" not in msg.lower():
            raise
if server is None:
    raise SystemExit("all serve configs OOM'd - lower gpu_memory_utilization further")

_alias_map = {a: TARGET_MODEL_NAME for a in (declared_model, TARGET_MODEL_NAME) if a}
# the agent resolves its model from agent.yaml, not from our alias list: include it or
# every rep dies with "model ... does not exist" (seen 2026-10-31, E4B run, 0/129)
import re as _re2
_m2 = _re2.search(r'(?m)^\s*model:\s*(\S+)', (AGENT / 'agent.yaml').read_text(encoding='utf-8'))
_declared_in_pack = _m2.group(1) if _m2 else None
print('agent.yaml declares:', _declared_in_pack, '| serving:', TARGET_MODEL_NAME)
if _declared_in_pack:
    _alias_map[_declared_in_pack] = TARGET_MODEL_NAME
print('alias map:', _alias_map)
models = server.create_model_registry(aliases=_alias_map,
                                      model_prefix='openai/', api_key='EMPTY')
os.environ['OPENAI_BASE_URL'] = server.base_url
os.environ['OPENAI_API_KEY'] = 'EMPTY'
print("OPENAI_BASE_URL:", os.environ['OPENAI_BASE_URL'])

# Probe the exact model ids the API advertises, then prove a real completion works.
# A name mismatch here costs 2 s instead of a whole eval run.
import json as _json, urllib.request as _req
_ids = [m['id'] for m in _json.load(_req.urlopen(server.base_url.rstrip('/') + '/models'))['data']]
print('served model ids:', _ids)
if TARGET_MODEL_NAME not in _ids:
    raise SystemExit(f'served ids {_ids} do not include {TARGET_MODEL_NAME!r}')
# probe the id the AGENT will actually ask for, not just the one we advertised
for _probe_model in {_declared_in_pack, TARGET_MODEL_NAME} - {None}:
    _body = _json.dumps({'model': _probe_model, 'max_tokens': 8,
                         'messages': [{'role': 'user', 'content': 'Reply with OK'}]}).encode()
    _r = _req.Request(server.base_url.rstrip('/') + '/chat/completions', data=_body,
                      headers={'Content-Type': 'application/json', 'Authorization': 'EMPTY'})
    _resp = _json.load(_req.urlopen(_r))
    print('SMOKE OK (%s) ->' % _probe_model, repr(_resp['choices'][0]['message'].get('content'))[:60], flush=True)


In [ ]:

# ---------- 3. Evaluate ALL 129 tasks (resumable, chunked so one crash is not fatal) ----------
OUT = Path("/kaggle/working/results")
OUT.mkdir(parents=True, exist_ok=True)
res = OUT / "task_results.jsonl"          # rewritten by EVERY invocation: holds only the current chunk
accum = OUT / "task_results.all.jsonl"    # our durable accumulator across chunks / restarts

# --- mirror the GRADER: it reads eval_config.yaml (timeout_seconds, max_tool_calls,
# max_time_minutes, max_turns) and enforces those caps. The CLI defaults are
# 60 minutes / no call cap, which is NOT the board. Without this, our pool score
# is measured in a regime the grader never runs.
import yaml as _yaml
_cfg = {}
_ec = Path(AGENT) / "eval_config.yaml"
if _ec.exists():
    _raw = _yaml.safe_load(_ec.read_text()) or {}
    _cfg = _raw.get("evaluation", _raw) or {}
CAPS = []
for _k, _flag in (("max_tool_calls", "--max-tool-calls"),
                  ("max_time_minutes", "--max-time-minutes"),
                  ("max_turns", "--max-turns")):
    if _cfg.get(_k) is not None:
        CAPS += [_flag, str(_cfg[_k])]
if os.environ.get("BOARD_CAPS", "1") == "0":
    CAPS = []
    print("WARNING: BOARD_CAPS=0 -> running in the harness default regime (60 min, no call cap)")
print("board caps:", CAPS or "(none)", flush=True)

rows_all = []
if accum.exists():
    rows_all = [json.loads(l) for l in open(accum) if l.strip()]
done = {r["instance_id"] for r in rows_all}
all_ids = [json.loads(l)["instance_id"] for l in open(TASKS) if l.strip()]
todo = [t for t in all_ids if t not in done]
print(f"todo {len(todo)}/{len(all_ids)} (resumed {len(done)})")

CONC = "10"
CHUNK = 20
for i in range(0, len(todo), CHUNK):
    batch = todo[i:i + CHUNK]
    cmd = [sys.executable, "-m", "swegemma", "eval",
           "--tasks", str(TASKS), "--snapshots-dir", str(SNAPS),
           "--results-dir", str(OUT), "--submission-dir", str(AGENT),
           "--sandbox", "subprocess", "--concurrency", CONC, "--display", "quiet"] + CAPS
    for t in batch:
        cmd += ["--task-id", t]
    print(f"--- chunk {i//CHUNK+1}: {len(batch)} tasks ---", flush=True)
    # PER-TASK LIVE LINES (print-only change; scoring path untouched): poll the results file while the
    # chunk runs and print each task the moment its row lands, then a final sweep after the chunk exits.
    import time as _t
    _printed = {r["instance_id"] for r in rows_all}
    def _emit_new():
        if not res.exists():
            return
        for _l in open(res):
            if not _l.strip():
                continue
            try:
                _r = json.loads(_l)
            except Exception:
                continue
            _id = _r.get("instance_id")
            if _id in _printed or _id not in batch:
                continue
            _printed.add(_id)
            print(f"  {_id:<16} -> resolved={bool(_r.get('resolved'))} duration={_r.get('duration_seconds')}s "
                  f"tool_calls={_r.get('tool_calls')} patch_chars={_r.get('agent_patch_size')} "
                  f"test_rc={_r.get('test_exit_code')} error={str(_r.get('error') or '')[:90]}", flush=True)
    _proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
    while _proc.poll() is None:
        _t.sleep(15)
        _emit_new()
    _so, _se = _proc.communicate()
    _emit_new()
    p = type("P", (), {"returncode": _proc.returncode, "stdout": _so, "stderr": _se})
    print("  rc:", p.returncode, (p.stdout or "")[-300:], (p.stderr or "")[-300:], flush=True)

    # harvest this chunk's rows into the accumulator before the next chunk destroys them
    seen = {r["instance_id"] for r in rows_all}
    if res.exists():
        for l in open(res):
            if l.strip():
                r = json.loads(l)
                if r["instance_id"] not in seen:
                    rows_all.append(r)
                    seen.add(r["instance_id"])
    with open(accum, "w") as f:
        for r in rows_all:
            f.write(json.dumps(r) + "\n")
    k_chunk = sum(int(bool(r.get("resolved"))) for r in rows_all)
    print(f"  cumulative: {k_chunk}/{len(rows_all)}", flush=True)

rows = rows_all
k = sum(int(bool(r.get("resolved"))) for r in rows)
print(f"\n=== ca5-31B FULL POOL: {k}/{len(rows)}" + (f" = {k/len(rows):.3f}" if rows else "") + " ===")
print("refs: 31B-v10 34/129 | E4B ca5 15/129 & 10/129 | off12(12B) pending")
by = {}
for r in rows:
    repo = r.get("repo") or r["instance_id"].split("_")[0]
    d = by.setdefault(repo, [0, 0]); d[0] += int(bool(r.get("resolved"))); d[1] += 1
for repo, (kk, nn) in sorted(by.items()):
    print(f"  {repo:<22} {kk:>3}/{nn:<3}")
with open("/kaggle/working/summary.json", "w") as f:
    json.dump({"resolved": k, "total": len(rows), "rate": (k / len(rows) if rows else 0),
               "by_repo": by, "prompt_sha": sha}, f, indent=2)
print("wrote /kaggle/working/summary.json")
